## Speed Ethogram Raster + Cumulative Probability
Panel (c)-style figure: per-trial speed heatmap and cumulative escape/freeze probability.
Load pre-computed results from the pipeline notebook.

In [1]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import os
from matplotlib import pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from scipy.ndimage import gaussian_filter1d

PROJECT_PATH = '/media/arnab/My Book Duo/Data/hoylab_projects/virtualcricket_loom/'

RESULT_CONFIGS = [
    {
        'save_dir': os.path.join(PROJECT_PATH, 'analysis_results_all_looms'),
        'label': 'all_looms',
        'title_suffix': '(all looms)',
    },
    {
        'save_dir': os.path.join(PROJECT_PATH, 'analysis_results_120s'),
        'label': '120s',
        'title_suffix': '(first 120s)',
    },
]

print('Configured result directories:')
for cfg in RESULT_CONFIGS:
    exists = os.path.exists(os.path.join(cfg['save_dir'], 'results.npz'))
    print(f"  {cfg['label']}: {cfg['save_dir']} [{'found' if exists else 'NOT FOUND'}]")

Configured result directories:


  all_looms: /media/arnab/My Book Duo/Data/hoylab_projects/virtualcricket_loom/analysis_results_all_looms [found]
  120s: /media/arnab/My Book Duo/Data/hoylab_projects/virtualcricket_loom/analysis_results_120s [found]


In [2]:
def build_speed_matrix(results, group_name, pad_onset, display_frames, shelter_dist_threshold=100):
    """Collect per-trial speed arrays (including pre-loom baseline) into a 2D matrix.
    After the mouse reaches shelter, speed is set to NaN."""
    speeds = []
    infos = []
    escape_latencies = []
    
    for vf in results[group_name]:
        trial_info = results[group_name][vf]['trial_info']
        speed_trials = results[group_name][vf]['mouse_speed_trialwise']
        shelter_dist_trials = results[group_name][vf]['mouse_shelter_distance_trialwise']
        esc_lat = results[group_name][vf].get('escape_latency_trialwise', [])
        
        valid_idx = 0
        for i, ti in enumerate(trial_info):
            if ti == 0:
                continue
            if valid_idx >= len(speed_trials):
                break
            
            spd = np.copy(speed_trials[valid_idx])
            
            # Blank out frames after mouse reaches shelter
            if valid_idx < len(shelter_dist_trials):
                shelter_dist = shelter_dist_trials[valid_idx]
                shelter_arrivals = np.where(shelter_dist[pad_onset:] <= shelter_dist_threshold)[0]
                if len(shelter_arrivals) > 0:
                    cutoff = pad_onset + shelter_arrivals[0]
                    spd[cutoff:] = np.nan
            
            # Use full trial starting from frame 0 (includes pre-loom baseline)
            row = np.full(display_frames, np.nan)
            n = min(len(spd), display_frames)
            row[:n] = spd[:n]
            
            speeds.append(row)
            infos.append(ti)
            
            if ti == 1 and valid_idx < len(esc_lat):
                escape_latencies.append(esc_lat[valid_idx])
            else:
                escape_latencies.append(np.nan)
            
            valid_idx += 1
    
    if len(speeds) == 0:
        return np.array([]).reshape(0, display_frames), np.array([]), np.array([])
    
    return np.array(speeds), np.array(infos), np.array(escape_latencies)


def get_freeze_onset_frame(speed_arr, pad_onset, freeze_speed_threshold=2.5, min_frames=15, sigma=1.0):
    """Find freeze onset frame in post-loom portion of speed array (same logic as detect_freeze)."""
    post_loom = speed_arr[pad_onset:]
    smooth = np.copy(post_loom)
    valid = ~np.isnan(smooth)
    if not np.any(valid):
        return np.nan
    smooth[valid] = gaussian_filter1d(smooth[valid], sigma=sigma)
    mask = smooth < freeze_speed_threshold
    
    count = 0
    for i, m in enumerate(mask):
        if m:
            count += 1
            if count >= min_frames:
                return i - min_frames + 1
        else:
            count = 0
    return np.nan


def get_escape_onset_frame(speed_arr, pad_onset, escape_speed_threshold=5.0, sigma=1.0):
    """Find escape onset frame in post-loom portion of speed array."""
    post_loom = speed_arr[pad_onset:]
    smooth = np.copy(post_loom)
    valid = ~np.isnan(smooth)
    if not np.any(valid):
        return np.nan
    smooth[valid] = gaussian_filter1d(smooth[valid], sigma=sigma)
    indices = np.where(smooth > escape_speed_threshold)[0]
    if len(indices) == 0:
        return np.nan
    return indices[0]


def get_onset_times(speed_matrix, infos, escape_latencies, pad_onset, fps):
    """Compute escape and freeze onset times in seconds (relative to loom onset)."""
    escape_onsets = []
    freeze_onsets = []
    
    for i in range(len(infos)):
        if infos[i] == 1:
            lat = escape_latencies[i]
            onset_sec = None
            # Tier 1: use stored escape_latency if valid and post-loom
            if not np.isnan(lat) and lat > pad_onset:
                onset_sec = (lat - pad_onset) / fps
            # Tier 2: detect from speed trace
            if onset_sec is None:
                frame = get_escape_onset_frame(speed_matrix[i], pad_onset)
                if not np.isnan(frame):
                    onset_sec = frame / fps
            # Tier 3: count at loom onset
            if onset_sec is None:
                onset_sec = 0.0
            escape_onsets.append(onset_sec)
        elif infos[i] == 2:
            frame = get_freeze_onset_frame(speed_matrix[i], pad_onset)
            if not np.isnan(frame):
                freeze_onsets.append(frame / fps)
    
    return sorted(escape_onsets), sorted(freeze_onsets)


def sort_trials(speed_matrix, infos, escape_latencies, pad_onset):
    """Sort rows: escapes (by onset), freezes (by onset), no-response."""
    escape_idx, freeze_idx, noresp_idx = [], [], []
    escape_keys, freeze_keys = [], []
    
    for i in range(len(infos)):
        if infos[i] == 1:
            lat = escape_latencies[i]
            key = (lat - pad_onset) if not np.isnan(lat) else 1e9
            escape_idx.append(i)
            escape_keys.append(key)
        elif infos[i] == 2:
            frame = get_freeze_onset_frame(speed_matrix[i], pad_onset)
            freeze_idx.append(i)
            freeze_keys.append(frame if not np.isnan(frame) else 1e9)
        else:
            noresp_idx.append(i)
    
    order = (
        [escape_idx[j] for j in np.argsort(escape_keys)] +
        [freeze_idx[j] for j in np.argsort(freeze_keys)] +
        noresp_idx
    )
    return order


def make_custom_cmap():
    """Red (0) -> black (10) -> green (40) diverging colormap."""
    colors = [
        (0.0, (0.85, 0.15, 0.15)),
        (0.25, (0.0, 0.0, 0.0)),
        (1.0, (0.15, 0.65, 0.15)),
    ]
    cmap = LinearSegmentedColormap.from_list('rbg_speed', colors, N=256)
    cmap.set_bad('white')
    return cmap


TRIAL_TYPE_COLORS = {
    1: np.array([0x47, 0x89, 0x00]) / 255,  # escape — green
    2: np.array([0xcc, 0x33, 0x33]) / 255,  # freeze — red
    3: np.array([0x88, 0x88, 0x88]) / 255,  # no-response — grey
}

def build_trial_type_strip(sorted_infos, width=4):
    """Return an (n_trials, width, 3) RGB array color-coded by trial type."""
    n = len(sorted_infos)
    strip = np.zeros((n, width, 3))
    for i, ti in enumerate(sorted_infos):
        strip[i] = TRIAL_TYPE_COLORS.get(ti, TRIAL_TYPE_COLORS[3])
    return strip

print('Helper functions defined.')

Helper functions defined.


In [3]:
def plot_ethogram_panel(speed_matrix, infos, escape_latencies, group_name,
                        cmap, vmin, vmax, pad_onset, fps,
                        pre_loom_s, post_loom_s):
    """Create a 2-panel figure: speed raster + cumulative probability."""
    n_trials = speed_matrix.shape[0]
    if n_trials == 0:
        print(f'  {group_name}: no valid trials, skipping.')
        return None
    
    # Sort trials
    order = sort_trials(speed_matrix, infos, escape_latencies, pad_onset)
    sorted_matrix = speed_matrix[order]
    sorted_infos = infos[order]
    
    # Get onset times for CDF
    escape_onsets, freeze_onsets = get_onset_times(
        speed_matrix, infos, escape_latencies, pad_onset, fps)
    n_valid = int(np.sum(infos > 0))
    
    # Figure layout: raster + CDF
    fig = plt.figure(figsize=(3.5, 4.0), dpi=300)
    gs = fig.add_gridspec(2, 1, height_ratios=[0.6, 0.4], hspace=0.35)
    
    ax_raster = fig.add_subplot(gs[0])
    ax_cdf = fig.add_subplot(gs[1])
    
    # Raster — x-axis from -pre_loom_s to +post_loom_s
    extent = [-pre_loom_s, post_loom_s, n_trials - 0.5, -0.5]
    im = ax_raster.imshow(sorted_matrix, aspect='auto', cmap=cmap,
                          vmin=vmin, vmax=vmax, interpolation='nearest',
                          extent=extent)
    ax_raster.axvline(0, color='white', linewidth=0.8, linestyle='--', alpha=0.7)
    ax_raster.set_ylabel(f'{n_trials} trials', fontsize=8)
    ax_raster.set_yticks([])
    ax_raster.set_xlabel('time (s)', fontsize=8)
    ax_raster.set_xticks(np.arange(int(-pre_loom_s), int(post_loom_s) + 1))
    ax_raster.tick_params(labelsize=7)
    
    # Horizontal lines between trial-type groups
    n_escape = int(np.sum(sorted_infos == 1))
    n_freeze = int(np.sum(sorted_infos == 2))
    for boundary in [n_escape, n_escape + n_freeze]:
        if 0 < boundary < n_trials:
            ax_raster.axhline(boundary - 0.5, color='white', linewidth=1.0)
    
    # Trial-type strip as inset flush against right edge
    ax_strip = inset_axes(ax_raster, width='2.5%', height='100%', loc='right',
                          borderpad=0)
    strip = build_trial_type_strip(sorted_infos)
    ax_strip.imshow(strip, aspect='auto', interpolation='nearest',
                    extent=[0, 1, n_trials - 0.5, -0.5])
    ax_strip.set_xticks([])
    ax_strip.set_yticks([])
    for spine in ax_strip.spines.values():
        spine.set_visible(False)
    
    # Colorbar
    cbar = fig.colorbar(im, ax=ax_raster, fraction=0.03, pad=0.06)
    cbar.set_ticks([0, 10, 40])
    cbar.set_label('Speed (cm/s)', fontsize=7)
    cbar.ax.tick_params(labelsize=6)
    
    # CDF — x-axis matches heatmap (-pre_loom_s to +post_loom_s)
    t_grid = np.linspace(-pre_loom_s, post_loom_s, 200)
    
    if n_valid > 0:
        if escape_onsets:
            esc_cdf = np.searchsorted(escape_onsets, t_grid) / n_valid
            ax_cdf.plot(t_grid, esc_cdf, color='#478900', linewidth=1.5, label='escape')
        
        if freeze_onsets:
            frz_cdf = np.searchsorted(freeze_onsets, t_grid) / n_valid
            ax_cdf.plot(t_grid, frz_cdf, color='#cc3333', linewidth=1.5, label='freeze')
    
    ax_cdf.axvline(0, color='grey', linewidth=0.8, linestyle='--', alpha=0.5)
    ax_cdf.set_xlim(-pre_loom_s, post_loom_s)
    ax_cdf.set_ylim(0, 1)
    ax_cdf.set_xlabel('time (s)', fontsize=8)
    ax_cdf.set_ylabel('cumulative prob.', fontsize=8)
    ax_cdf.set_xticks(np.arange(int(-pre_loom_s), int(post_loom_s) + 1))
    ax_cdf.tick_params(labelsize=7)
    ax_cdf.spines['top'].set_visible(False)
    ax_cdf.spines['right'].set_visible(False)
    ax_cdf.legend(fontsize=6, frameon=False)
    
    title = group_name.replace('_', ' ').replace('shelter', '').strip()
    fig.suptitle(title, fontsize=9, y=0.98)
    fig.subplots_adjust(hspace=0.3, top=0.93)
    
    return fig

print('Plot function defined.')

Plot function defined.


In [4]:
def plot_combined_ethogram(results, experience, cmap, vmin, vmax,
                           pad_onset, fps, pre_loom_s, post_loom_s, display_frames):
    """Create a 2x2 figure (age x sex), each cell has raster + CDF."""
    ages = ['adolescent', 'adult']
    sexes = ['males', 'females']
    
    fig = plt.figure(figsize=(7, 8), dpi=300)
    outer_gs = fig.add_gridspec(2, 2, hspace=0.45, wspace=0.35)
    
    for row, age in enumerate(ages):
        for col, sex in enumerate(sexes):
            gn = f'{experience}_{age}_{sex}_shelter'
            if gn not in results:
                continue
            
            speed_mat, infos, esc_lat = build_speed_matrix(results, gn, pad_onset, display_frames)
            n_trials = speed_mat.shape[0]
            if n_trials == 0:
                continue
            
            order = sort_trials(speed_mat, infos, esc_lat, pad_onset)
            sorted_matrix = speed_mat[order]
            sorted_infos = infos[order]
            escape_onsets, freeze_onsets = get_onset_times(
                speed_mat, infos, esc_lat, pad_onset, fps)
            n_valid = int(np.sum(infos > 0))
            
            inner_gs = outer_gs[row, col].subgridspec(2, 1, height_ratios=[0.6, 0.4], hspace=0.4)
            ax_raster = fig.add_subplot(inner_gs[0])
            ax_cdf = fig.add_subplot(inner_gs[1])
            
            # Raster
            extent = [-pre_loom_s, post_loom_s, n_trials - 0.5, -0.5]
            im = ax_raster.imshow(sorted_matrix, aspect='auto', cmap=cmap,
                                  vmin=vmin, vmax=vmax, interpolation='nearest',
                                  extent=extent)
            ax_raster.axvline(0, color='white', linewidth=0.8, linestyle='--', alpha=0.7)
            ax_raster.set_ylabel(f'{n_trials} trials', fontsize=7)
            ax_raster.set_yticks([])
            ax_raster.set_xlabel('time (s)', fontsize=7)
            ax_raster.set_xticks(np.arange(int(-pre_loom_s), int(post_loom_s) + 1))
            ax_raster.tick_params(labelsize=6)
            ax_raster.set_title(f'{age} {sex}', fontsize=8)
            
            # Horizontal lines between trial-type groups
            n_escape = int(np.sum(sorted_infos == 1))
            n_freeze = int(np.sum(sorted_infos == 2))
            for boundary in [n_escape, n_escape + n_freeze]:
                if 0 < boundary < n_trials:
                    ax_raster.axhline(boundary - 0.5, color='white', linewidth=1.0)
            
            # Trial-type strip as inset flush against right edge
            ax_strip = inset_axes(ax_raster, width='2.5%', height='100%', loc='right',
                                  borderpad=0)
            strip = build_trial_type_strip(sorted_infos)
            ax_strip.imshow(strip, aspect='auto', interpolation='nearest',
                            extent=[0, 1, n_trials - 0.5, -0.5])
            ax_strip.set_xticks([])
            ax_strip.set_yticks([])
            for spine in ax_strip.spines.values():
                spine.set_visible(False)
            
            cbar = fig.colorbar(im, ax=ax_raster, fraction=0.03, pad=0.06)
            cbar.set_ticks([0, 10, 40])
            cbar.set_label('Speed (cm/s)', fontsize=6)
            cbar.ax.tick_params(labelsize=5)
            
            # CDF — x-axis matches heatmap (-pre_loom_s to +post_loom_s)
            t_grid = np.linspace(-pre_loom_s, post_loom_s, 200)
            if n_valid > 0:
                if escape_onsets:
                    esc_cdf = np.searchsorted(escape_onsets, t_grid) / n_valid
                    ax_cdf.plot(t_grid, esc_cdf, color='#478900', linewidth=1.5, label='escape')
                if freeze_onsets:
                    frz_cdf = np.searchsorted(freeze_onsets, t_grid) / n_valid
                    ax_cdf.plot(t_grid, frz_cdf, color='#cc3333', linewidth=1.5, label='freeze')
            
            ax_cdf.axvline(0, color='grey', linewidth=0.8, linestyle='--', alpha=0.5)
            ax_cdf.set_xlim(-pre_loom_s, post_loom_s)
            ax_cdf.set_ylim(0, 1)
            ax_cdf.set_xlabel('time (s)', fontsize=7)
            ax_cdf.set_ylabel('cumulative prob.', fontsize=7)
            ax_cdf.set_xticks(np.arange(int(-pre_loom_s), int(post_loom_s) + 1))
            ax_cdf.tick_params(labelsize=6)
            ax_cdf.spines['top'].set_visible(False)
            ax_cdf.spines['right'].set_visible(False)
            ax_cdf.legend(fontsize=5, frameon=False)
    
    fig.suptitle(experience, fontsize=11, y=0.99)
    return fig

print('Combined plot function defined.')

Combined plot function defined.


In [5]:
custom_cmap = make_custom_cmap()
inferno_cmap = plt.cm.inferno.copy()
inferno_cmap.set_bad('white')

for cfg in RESULT_CONFIGS:
    results_path = os.path.join(cfg['save_dir'], 'results.npz')
    meta_path = os.path.join(cfg['save_dir'], 'metadata.npz')
    if not os.path.exists(results_path):
        print(f"Skipping {cfg['label']}: results not found at {cfg['save_dir']}")
        continue

    results = np.load(results_path, allow_pickle=True)['results'].item()
    meta = np.load(meta_path, allow_pickle=True)
    group_names = list(meta['group_names'])
    analysis_params = meta['analysis_params'].item()

    pad_onset = analysis_params['pad_onset']
    fps = analysis_params['fps']
    loom_duration = analysis_params['loom_duration']
    pad_offset = analysis_params['pad_offset']

    pre_loom_s = pad_onset / fps
    post_loom_s = (loom_duration + pad_offset) / fps
    display_seconds = pre_loom_s + post_loom_s
    display_frames = int(display_seconds * fps)

    # Output folders
    svg_dir = os.path.join(os.getcwd(), f'figure_svgs_{cfg["label"]}')
    png_dir = os.path.join(os.getcwd(), f'figure_pngs_{cfg["label"]}')
    os.makedirs(svg_dir, exist_ok=True)
    os.makedirs(png_dir, exist_ok=True)

    print(f'\n{"="*60}')
    print(f'{cfg["label"].upper()}: pad_offset={pad_offset}, window=-{pre_loom_s:.1f}s to +{post_loom_s:.1f}s')
    print(f'  SVGs -> {svg_dir}')
    print(f'  PNGs -> {png_dir}')
    print(f'{"="*60}')

    for cmap, cmap_name in [(custom_cmap, 'custom'), (inferno_cmap, 'inferno')]:
        suffix = '' if cmap_name == 'custom' else '_inferno'
        for experience in ['naive', 'experienced']:
            fig = plot_combined_ethogram(results, experience, cmap, 0, 40,
                                         pad_onset, fps, pre_loom_s, post_loom_s, display_frames)
            fname = f'{experience}_speed_ethogram_combined{suffix}'
            fig.suptitle(f'{experience} {cfg["title_suffix"]}', fontsize=11, y=0.99)
            fig.savefig(os.path.join(svg_dir, f'{fname}.svg'), format='svg', bbox_inches='tight')
            fig.savefig(os.path.join(png_dir, f'{fname}.png'), format='png', bbox_inches='tight')
            plt.close(fig)
            print(f'  {fname}')

print('\nDone.')


ALL_LOOMS: pad_offset=105, window=-1.0s to +5.0s
  SVGs -> /home/arnab/Code/hoylab_repos/hoylab_analysis/virtual_cricket_loom_claude_cleaned/figure_svgs_all_looms
  PNGs -> /home/arnab/Code/hoylab_repos/hoylab_analysis/virtual_cricket_loom_claude_cleaned/figure_pngs_all_looms


  naive_speed_ethogram_combined


  experienced_speed_ethogram_combined


  naive_speed_ethogram_combined_inferno


  experienced_speed_ethogram_combined_inferno

120S: pad_offset=105, window=-1.0s to +5.0s
  SVGs -> /home/arnab/Code/hoylab_repos/hoylab_analysis/virtual_cricket_loom_claude_cleaned/figure_svgs_120s
  PNGs -> /home/arnab/Code/hoylab_repos/hoylab_analysis/virtual_cricket_loom_claude_cleaned/figure_pngs_120s


  naive_speed_ethogram_combined


  experienced_speed_ethogram_combined


  naive_speed_ethogram_combined_inferno


  experienced_speed_ethogram_combined_inferno

Done.


## Bar Plots: Velocity & Cricket Distance at Loom Onset (Escape vs Freeze vs No-Response)

In [6]:
def extract_loom_onset_values(results, group_name, pad_onset, fps, window_s=0.5):
    """Extract mean speed and cricket distance in a window before loom onset, grouped by mouse and trial type."""
    window_frames = int(window_s * fps)
    mouse_data = {}

    for vf in results[group_name]:
        mouse_id = vf.rsplit('_', 1)[0]
        trial_info = results[group_name][vf]['trial_info']
        speed_trials = results[group_name][vf]['mouse_speed_trialwise']
        dist_trials = results[group_name][vf]['mouse_cricket_distance_trialwise']

        valid_idx = 0
        for ti in trial_info:
            if ti == 0:
                continue
            if valid_idx >= len(speed_trials) or valid_idx >= len(dist_trials):
                break

            spd_arr = speed_trials[valid_idx]
            dst_arr = dist_trials[valid_idx]

            # Average over [pad_onset - window_frames, pad_onset)
            win_start = max(0, pad_onset - window_frames)
            win_end = min(pad_onset, len(spd_arr))

            if win_start < win_end:
                speed_val = np.nanmean(spd_arr[win_start:win_end])
                dist_val = np.nanmean(dst_arr[win_start:win_end])
            else:
                speed_val = np.nan
                dist_val = np.nan

            if mouse_id not in mouse_data:
                mouse_data[mouse_id] = {
                    'escape_speed': [], 'freeze_speed': [], 'noresp_speed': [],
                    'escape_distance': [], 'freeze_distance': [], 'noresp_distance': [],
                }

            if ti == 1:
                mouse_data[mouse_id]['escape_speed'].append(speed_val)
                mouse_data[mouse_id]['escape_distance'].append(dist_val)
            elif ti == 2:
                mouse_data[mouse_id]['freeze_speed'].append(speed_val)
                mouse_data[mouse_id]['freeze_distance'].append(dist_val)
            elif ti == 3:
                mouse_data[mouse_id]['noresp_speed'].append(speed_val)
                mouse_data[mouse_id]['noresp_distance'].append(dist_val)

            valid_idx += 1

    return mouse_data


def aggregate_per_mouse(mouse_data, variable):
    """Compute per-mouse mean for a given variable key (e.g. 'escape_speed')."""
    values = []
    for mid in mouse_data:
        trials = mouse_data[mid].get(variable, [])
        trials = [v for v in trials if not np.isnan(v)]
        if trials:
            values.append(np.mean(trials))
    return values

print('Data extraction functions defined.')

Data extraction functions defined.


In [7]:
ESCAPE_COLOR = '#478900'
FREEZE_COLOR = '#cc3333'
NORESP_COLOR = '#888888'
BAR_LABELS = ['escape', 'freeze', 'no resp.']
BAR_COLORS = [ESCAPE_COLOR, FREEZE_COLOR, NORESP_COLOR]


def plot_bar_comparison(results, experience, var_prefix, pad_onset, fps, ylabel, title_suffix):
    """Create 2x2 bar plots (age x sex) comparing escape/freeze/no-response at loom onset."""
    ages = ['adolescent', 'adult']
    sexes = ['males', 'females']
    keys = [f'escape_{var_prefix}', f'freeze_{var_prefix}', f'noresp_{var_prefix}']

    fig, axes = plt.subplots(2, 2, figsize=(6, 5), dpi=300, sharey=True)

    for row, age in enumerate(ages):
        for col, sex in enumerate(sexes):
            ax = axes[row, col]
            gn = f'{experience}_{age}_{sex}_shelter'
            if gn not in results:
                ax.set_title(f'{age} {sex}', fontsize=8)
                ax.set_visible(False)
                continue

            mouse_data = extract_loom_onset_values(results, gn, pad_onset, fps)
            group_vals = [aggregate_per_mouse(mouse_data, k) for k in keys]

            positions = np.arange(len(keys))
            bar_width = 0.5

            for i, (vals, color, label) in enumerate(zip(group_vals, BAR_COLORS, BAR_LABELS)):
                if vals:
                    mean = np.mean(vals)
                    sem = np.std(vals) / np.sqrt(len(vals)) if len(vals) > 1 else 0
                    ax.bar(positions[i], mean, bar_width, color=color, alpha=0.4,
                           edgecolor=color, linewidth=0.8, label=label)
                    ax.errorbar(positions[i], mean, yerr=sem, fmt='none',
                                ecolor='black', capsize=3, linewidth=0.8)
                    jitter = np.random.default_rng(42).uniform(-0.12, 0.12, len(vals))
                    ax.scatter(positions[i] + jitter, vals, s=15, facecolors='none',
                               edgecolors=color, linewidth=0.7, alpha=0.8, zorder=5)

            ax.set_xticks(positions)
            ax.set_xticklabels(BAR_LABELS, fontsize=6)
            ax.set_title(f'{age} {sex}', fontsize=8)
            ax.tick_params(labelsize=6)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            if col == 0:
                ax.set_ylabel(ylabel, fontsize=7)

    fig.suptitle(f'{experience} — {title_suffix}', fontsize=10, y=0.99)
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    return fig

print('Bar plot function defined.')

Bar plot function defined.


In [8]:
VARIABLES = [
    ('speed', 'mean speed 0.5s pre-loom (cm/s)', 'speed before loom onset'),
    ('distance', 'mean cricket distance 0.5s pre-loom', 'distance from cricket before loom onset'),
]

for cfg in RESULT_CONFIGS:
    results_path = os.path.join(cfg['save_dir'], 'results.npz')
    meta_path = os.path.join(cfg['save_dir'], 'metadata.npz')
    if not os.path.exists(results_path):
        print(f"Skipping {cfg['label']}: results not found")
        continue

    results = np.load(results_path, allow_pickle=True)['results'].item()
    meta = np.load(meta_path, allow_pickle=True)
    analysis_params = meta['analysis_params'].item()
    pad_onset = analysis_params['pad_onset']
    fps = analysis_params['fps']

    svg_dir = os.path.join(os.getcwd(), f'figure_svgs_{cfg["label"]}')
    png_dir = os.path.join(os.getcwd(), f'figure_pngs_{cfg["label"]}')
    os.makedirs(svg_dir, exist_ok=True)
    os.makedirs(png_dir, exist_ok=True)

    print(f'\n{"="*60}')
    print(f'BAR PLOTS — {cfg["label"].upper()}')
    print(f'{"="*60}')

    for var_prefix, ylabel, title_suffix in VARIABLES:
        for experience in ['naive', 'experienced']:
            fig = plot_bar_comparison(results, experience, var_prefix, pad_onset, fps,
                                       ylabel, f'{title_suffix} {cfg["title_suffix"]}')
            fname = f'{experience}_{var_prefix}_at_loom_onset'
            fig.savefig(os.path.join(svg_dir, f'{fname}.svg'), format='svg', bbox_inches='tight')
            fig.savefig(os.path.join(png_dir, f'{fname}.png'), format='png', bbox_inches='tight')
            plt.close(fig)
            print(f'  {fname}')

print('\nDone.')


BAR PLOTS — ALL_LOOMS


/tmp/ipykernel_8219/3375026402.py:28: RuntimeWarning: Mean of empty slice
  dist_val = np.nanmean(dst_arr[win_start:win_end])
/tmp/ipykernel_8219/3375026402.py:28: RuntimeWarning: Mean of empty slice
  dist_val = np.nanmean(dst_arr[win_start:win_end])
/tmp/ipykernel_8219/3375026402.py:28: RuntimeWarning: Mean of empty slice
  dist_val = np.nanmean(dst_arr[win_start:win_end])
/tmp/ipykernel_8219/3375026402.py:28: RuntimeWarning: Mean of empty slice
  dist_val = np.nanmean(dst_arr[win_start:win_end])


  naive_speed_at_loom_onset


/tmp/ipykernel_8219/3375026402.py:28: RuntimeWarning: Mean of empty slice
  dist_val = np.nanmean(dst_arr[win_start:win_end])
/tmp/ipykernel_8219/3375026402.py:28: RuntimeWarning: Mean of empty slice
  dist_val = np.nanmean(dst_arr[win_start:win_end])
/tmp/ipykernel_8219/3375026402.py:28: RuntimeWarning: Mean of empty slice
  dist_val = np.nanmean(dst_arr[win_start:win_end])
/tmp/ipykernel_8219/3375026402.py:28: RuntimeWarning: Mean of empty slice
  dist_val = np.nanmean(dst_arr[win_start:win_end])


  experienced_speed_at_loom_onset


...

  experienced_speed_at_loom_onset


/tmp/ipykernel_8219/3375026402.py:28: RuntimeWarning: Mean of empty slice
  dist_val = np.nanmean(dst_arr[win_start:win_end])
/tmp/ipykernel_8219/3375026402.py:28: RuntimeWarning: Mean of empty slice
  dist_val = np.nanmean(dst_arr[win_start:win_end])
/tmp/ipykernel_8219/3375026402.py:28: RuntimeWarning: Mean of empty slice
  dist_val = np.nanmean(dst_arr[win_start:win_end])


  naive_distance_at_loom_onset


/tmp/ipykernel_8219/3375026402.py:28: RuntimeWarning: Mean of empty slice
  dist_val = np.nanmean(dst_arr[win_start:win_end])
/tmp/ipykernel_8219/3375026402.py:28: RuntimeWarning: Mean of empty slice
  dist_val = np.nanmean(dst_arr[win_start:win_end])
/tmp/ipykernel_8219/3375026402.py:28: RuntimeWarning: Mean of empty slice
  dist_val = np.nanmean(dst_arr[win_start:win_end])
/tmp/ipykernel_8219/3375026402.py:28: RuntimeWarning: Mean of empty slice
  dist_val = np.nanmean(dst_arr[win_start:win_end])


  experienced_distance_at_loom_onset

Done.
